# 02 — What does a VLM learn?

There is no single VLM objective. Contrastive image-text pretraining (CLIP-style) makes matching pairs close and mismatches far apart in a shared embedding space. Generative pretraining predicts text tokens conditioned on image features. Image-text matching, captioning, OCR, grounding, and instruction-following objectives can be combined in stages. Dataset quality, negative construction, masking, and data mixture matter as much as the loss name.

## Symmetric contrastive loss (InfoNCE)

For a batch of N paired images and captions, normalize both embeddings, form an `N x N` similarity matrix, divide by a learned temperature, and classify the diagonal pairing in both directions. This in-batch negative formulation assumes each diagonal pair is the intended match; duplicate/near-duplicate examples can create false negatives.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

torch.manual_seed(11)
n, dim = 6, 16
image_z = F.normalize(torch.randn(n, dim), dim=-1)
text_z = F.normalize(image_z + 0.25 * torch.randn(n, dim), dim=-1)
log_temperature = torch.tensor(np.log(1 / 0.07), requires_grad=True)
logits = log_temperature.exp() * image_z @ text_z.T
targets = torch.arange(n)
loss_i = F.cross_entropy(logits, targets)
loss_t = F.cross_entropy(logits.T, targets)
loss = (loss_i + loss_t) / 2
print('similarities:', tuple(logits.shape), 'loss:', float(loss))
print('image->text top-1:', int((logits.argmax(1) == targets).sum()), '/', n)
loss.backward()
print('temperature gradient:', float(log_temperature.grad))

## Autoregressive caption / answer loss

Given prompt tokens and image features, a causal decoder predicts the next token. Teacher forcing supplies the true previous token at each training step. Shift input and target by one: the logit at position `t` predicts target token `t+1`. Ignore padding and prompt positions that should not contribute to the loss. The logits are `[B,T,V]`; labels are `[B,T]`.

In [ ]:
B, T, V = 2, 5, 9
logits = torch.randn(B, T, V, requires_grad=True)
tokens = torch.tensor([[1, 3, 4, 2, 0], [1, 5, 6, 7, 2]])
labels = tokens[:, 1:].contiguous()
prediction_logits = logits[:, :-1, :].contiguous()
pad_id = 0
token_loss = F.cross_entropy(prediction_logits.view(-1, V), labels.view(-1), ignore_index=pad_id)
token_loss.backward()
print('shifted logits:', tuple(prediction_logits.shape), 'labels:', tuple(labels.shape), 'loss:', float(token_loss))

## Grounding and alignment checks

A low average loss does not prove correct grounding. Useful task-specific objectives/metrics include image-text retrieval recall, localized box/point error, OCR exact match, VQA answer accuracy, and calibration. If generated text mentions an object absent from the image, that is a hallucination even when it is fluent. Data can also leak answers through filenames, alt text, duplicates, or split contamination.

**Exercises:** (1) Make one caption embedding a deliberate false negative. (2) Compare one-way and symmetric contrastive loss. (3) Set some labels to `-100` and explain why prompt tokens should or should not be supervised for your chosen task. (4) Check that a causal mask prevents looking at the answer token.